In [0]:
# Databricks notebook source
# =========================================================================
# KROK 1: KONTROLA ŚRODOWISKA I LIVE-RELOAD (Deweloperski Best Practice)
# =========================================================================
%load_ext autoreload
%autoreload 2

import sys
import os
import importlib

# Blokada generowania skompilowanego cache .pyc na klastrze chmurowym
sys.dont_write_bytecode = True

# Dynamiczne podpięcie katalogu głównego projektu do ścieżek Pythona
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Wymuszenie czystego przeładowania modułów z dysku do pamięci RAM klastra
import src.readers.dataframe_reader
import src.rules.physical_rules
import src.rules.query_rules
importlib.reload(src.readers.dataframe_reader)
importlib.reload(src.rules.physical_rules)
importlib.reload(src.rules.query_rules)

# =========================================================================
# KROK 2: IMPORTY CORE ENGINE ORAZ PEŁNEGO PAKIETU 10 REGUŁ
# =========================================================================
from src.auditor.engine import PerformanceEngine
from src.readers.dataframe_reader import DataFrameExplainReader
from src.policies.policy_manager import PolicyManager
from src.context.environment_provider import EnvironmentProvider
from src.suggestions.remediation_engine import RemediationEngine
from src.finops.cost_translator import CostTranslator
from src.reporters.console_reporter import ConsoleReporter

# Importy reguł fizycznych i infrastrukturalnych
from src.rules.physical_rules import (
    SmallFilesRule,          # PERF-001
    MissedBroadcastRule,     # PERF-002
    OverPartitioningRule,    # PERF-004
    DataSkewRule,            # PERF-006
    MissingOptimizationRule  # PERF-007
)

# Importy reguł logicznych oraz optymalizacji Catalyst
from src.rules.query_rules import (
    TypeCastingRule,         # PERF-003
    CartesianProductRule,    # PERF-005
    ExplodeAbuseRule,        # PERF-008
    RedundantScanRule,       # PERF-009
    NonVectorizedUdfRule     # PERF-010
)

# =========================================================================
# KROK 3: UNIWERSALNA FUNKCJA ORKIESTRUJĄCA (Zunifikowany Silnik)
# =========================================================================
def audit_production_process(
    context_name: str, 
    table_name: str = None, 
    df = None, 
    source_volume_path: str = None
):
    """
    Uruchamia pełną serię 10 testów diagnostycznych APM & FinOps 
    dla jednego konkretnego procesu przetwarzania danych.
    
    :param context_name: Unikalna nazwa identyfikująca proces (etykieta raportu)
    :param table_name: Nazwa tabeli/widoku w Unity Catalog (opcjonalna - pod Serverless)
    :param df: Instancja PySpark DataFrame badana w locie (opcjonalna)
    :param source_volume_path: Fizyczna ścieżka do katalogu/wolumenu plików (opcjonalna)
    """
    print(f"\n" + "═"*70)
    print(f"🎬 URUCHAMIAM INTEGRALNĄ DIAGNOSTYKĘ APM DLA: {context_name}")
    print(f"═"*70, flush=True)
    
    # 1. Inicjalizacja stosu technologicznego (Wstrzykiwanie zależności)
    policy_mgr = PolicyManager()
    env_prov = EnvironmentProvider(spark)
    remedy_eng = RemediationEngine()
    cost_trans = CostTranslator(policy_mgr.get_policy("finops"))
    reporter = ConsoleReporter()
    
    # 2. Definicja i konsolidacja kompletnego zestawu 10 detektywów
    complete_diagnostic_suite = [
        SmallFilesRule(),          # PERF-001
        MissedBroadcastRule(),     # PERF-002
        TypeCastingRule(),         # PERF-003
        OverPartitioningRule(),    # PERF-004
        CartesianProductRule(),    # PERF-005
        DataSkewRule(),            # PERF-006
        MissingOptimizationRule(), # PERF-007
        ExplodeAbuseRule(),        # PERF-008
        RedundantScanRule(),       # PERF-009
        NonVectorizedUdfRule()     # PERF-010
    ]
    
    # 3. Dynamiczne rozwiązywanie DataFrame, jeśli podano tylko nazwę tabeli
    resolved_df = df
    if resolved_df is None and table_name:
        try:
            resolved_df = spark.read.table(table_name)
        except Exception as e:
            print(f"⚠️  [PRE-FLIGHT-WARN] Nie można załadować tabeli {table_name}: {str(e)}")
            
    # 4. Konstrukcja polimorficznego czytnika planu wykonania
    reader = DataFrameExplainReader(
        spark=spark, 
        df=resolved_df, 
        table_name=table_name, 
        source_volume_path=source_volume_path
    )
    
    # 5. Montaż orkiestratora i odpalenie pełnej pętli audytowej
    engine = PerformanceEngine(
        reader=reader,
        rules=complete_diagnostic_suite,
        policy_manager=policy_mgr,
        env_provider=env_prov,
        remediation_engine=remedy_eng,
        cost_translator=cost_trans,
        reporter=reporter
    )
    
    engine.run_audit(context_name=context_name)
    print(f"🏁 Zakończono serię diagnostyczną dla: {context_name}\n" + "═"*70)
    audit_production_process(
    context_name="ETL-01-BESS-Telemetry",
    table_name="workspace.bronze.bess_telemetry_small_files",
    source_volume_path="/Volumes/workspace/default/weather_data/raw_source/bess_telemetry_raw"
)
audit_production_process(
    context_name="ETL-03-Station-Enrichment",
    table_name="workspace.bronze.enriched_telemetry_heavy_shuffle"
)

In [0]:
from pyspark.sql import functions as F

# Tworzymy badany proces w locie (np. potok PV z rzutowaniem typów)
df_pv_pipeline = spark.read.table("workspace.bronze.pv_metrics_string_nightmare") \
                           .withColumn("calculated_factor", F.explode(F.array(F.lit(1), F.lit(2))))

# Odpalamy diagnostykę podając czysty obiekt biznesowy
audit_production_process(
    context_name="ETL-02-PV-Pipeline-AdHoc",
    df=df_pv_pipeline
)